In [ ]:
import time
import json
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
from matplotlib.colors import Normalize
from PIL import Image

from pc import PS
from modules import ADC,DAC,CHIP
from cimCommand import CMD,CmdData,Packet
from cimCommand.singleCmdInfo import *

from util import plot_v_cond,plot_cond

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0)

# 灰度图

### 1. 处理图片

In [ ]:
image_path = "./images/yuquan.jpg"
image = Image.open(image_path)

gray_image = image.convert("L")
width, height = gray_image.size
left = (width - 256) / 2
top = (height - 256) / 2
right = (width + 256) / 2
bottom = (height + 256) / 2

if width < 256 or height < 256:
    resized_image = gray_image.resize((256, 256))
else:
    cropped_image = gray_image.crop((left, top, right, bottom))
    resized_image = cropped_image.resize((256, 256))

resized_image = np.array(resized_image)
plt.imshow(resized_image, cmap='gray')
plt.axis('off')
plt.show()

### 2.拟合tg和cond的线性映射

In [ ]:
# 3v,set脉宽1us,reset脉宽1us
tg_map = [1.4,1.5,1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3,2.4]
cond_map = [190,310,430,510,610,710,850,910,1030,1110,1190]
# 3v,set脉宽1us,reset脉宽10us
# tg_map = [1.4,1.5,1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3,2.4]
# cond_map = [190,310,430,510,610,710,850,910,1030,1110,1190]

# tg_map = [1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3]
# cond_map = [550,670,790,890,1010,1090,1170,1250]

tg_map = np.array(tg_map)
cond_map = np.array(cond_map)

slope,intercept = np.polyfit(tg_map, cond_map, 1)  # 返回斜率和截距

y_fit = slope * tg_map + intercept

plt.scatter(tg_map, cond_map, color='blue', label='Data points')  # 原始数据点
plt.plot(tg_map, y_fit, color='red', label=f'cond = {slope:.2f} * tg + {intercept:.2f}')  # 拟合直线
plt.xlabel('tg(v)')
plt.ylabel('cond(us)')
plt.legend()
plt.title('Linear Fit')
plt.show()

### 3.直接写对应值

In [ ]:
cond_min = slope * 1.2 + intercept
cond_max = slope * 2.2 + intercept

cond_target = cond_min + resized_image / 255 * (cond_max - cond_min)
cond_upper = cond_target + 25
cond_lower = cond_target - 25

img_tg = (cond_target - intercept)/slope

plot_cond(cond_target,np.min(cond_lower),np.max(cond_upper))
# print("上下界")
# plot_cond(cond_lower,np.min(cond_lower),np.max(cond_upper))
# plot_cond(cond_upper,np.min(cond_lower),np.max(cond_upper))

In [ ]:
def write_verify(write_time,img_cond_lower,img_cond_upper,tg_v,set_pulse_width,reset_pulse_width,root_path):
    if not os.path.exists(root_path):
        os.makedirs(root_path, exist_ok=True)
    vmin = np.min(img_cond_lower)
    vmax = np.max(img_cond_upper)
    need_read = np.ones((256,256),dtype=bool)
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    cond_sub_base = None
    for i in range(write_time):
        print(f"第{i}次写验证")
        voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        np.save(root_path+f"after_write_verify_time={int(i)}.npy", cond_sub_base)

        condition_reset = (cond_sub_base > img_cond_upper) & need_read
        condition_set = (cond_sub_base < img_cond_lower) & need_read
        need_read = condition_reset | condition_set
        
        path = root_path+f"{i}.png"
        plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)

        if i>0:
            if i<5:
                tg_v[condition_reset] -= 0.08
                tg_v[condition_set] += 0.08
            elif i<10:
                tg_v[condition_reset] -= 0.04
                tg_v[condition_set] += 0.04
            else:
                tg_v[condition_reset] -= 0.02
                tg_v[condition_set] += 0.02

        # reset的点
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=5,pulse_width=reset_pulse_width,set_device=False)
        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=tg_v,pulse_width=set_pulse_width,set_device=True)

        # set的点
        chip.write_point2(crossbar=condition_set,write_voltage=3,tg=tg_v,pulse_width=set_pulse_width,set_device=True)


    voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    np.save(root_path+f"after_write_verify_time={write_time}.npy", cond_sub_base)
    np.save(root_path+f"tg_v.npy", tg_v)

    condition_reset = (cond_sub_base > img_cond_upper) & need_read
    condition_set = (cond_sub_base < img_cond_lower) & need_read
    need_read = condition_reset | condition_set
    path = root_path+f"{write_time}.png"
    plot_cond(cond_sub_base,vmin,vmax,title=f"{i},need_reset={int(np.sum(condition_reset))},need_set={int(np.sum(condition_set))}",path = path)
    return cond_sub_base

In [ ]:
img_tg = (cond_target - intercept)/slope-0.1
write_time = 20
root_path = "./result/write_verify_12_22_8/"
cond_sub_base = write_verify(write_time=write_time,img_cond_lower=cond_lower,img_cond_upper=cond_upper,
                             tg_v=img_tg,set_pulse_width=1e-6,reset_pulse_width=1e-6,root_path=root_path)

In [ ]:
# cond_sub_base = np.load(root_path+f"after_write_verify_time={write_time}.npy")

need_read = np.ones((256,256))
voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)


bound = 100
interval = 1
bin_edges = np.linspace(-bound, bound, int((bound*2)/interval)+1)
data = (cond_sub_base-cond_target).flatten()
data[data<-bound]=-bound
data[data>bound]=bound

counts, bin_edges, _ = plt.hist(data, bins=bin_edges, color='blue', alpha=0.7, edgecolor='None')

plt.title(f"real_cond - target_cond")
plt.xlabel("cond(uS)")
plt.ylabel("Frequency")

plt.savefig(root_path+f"error.png")  # 保存为 PNG 格式
plt.show()

# 二值图片

In [ ]:
img = Image.open('../images/binary_256_256.png').convert('L')
img = img.resize((256, 256), Image.LANCZOS)

img = np.array(img)
img = np.where(img >= 128, 0, 1).astype(np.uint8)

# row,col = img.shape
# for i in range(row):
#     for j in range(col):
#         print(img[i,j],end=' ')
#     print('')

plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
def write_gray(img,write_times,start_tg_v = 1,delta_tg_v = 0.1,lower_bound=200,upper_bound=800,pulse_width = 1e-6):
    voltage_base = np.zeros((256,256))
    voltage = np.zeros((256,256))
    need_read = np.ones((256,256),dtype=bool)

    for i in range(write_times):
        tgv = start_tg_v+i*delta_tg_v
        voltage_base[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        voltage[need_read] = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[need_read]
        cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
        condition_reset = (cond_sub_base > lower_bound) & (img<0.5) & need_read
        condition_set = (cond_sub_base < upper_bound) & (img>0.5) & need_read

        plot_cond(cond_sub_base,vmax=1000,title=f"set_tgv={int(tgv*10)},need_set={int(np.sum(condition_set))}")

        chip.write_point2(crossbar=condition_reset,write_voltage=3,tg=5,pulse_width=pulse_width,set_device=False)
        chip.write_point2(crossbar=condition_set,write_voltage=3,tg=tgv,pulse_width=pulse_width,set_device=True)

    voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)
    voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)
    cond_sub_base = chip.voltage_to_cond(voltage-voltage_base)
    condition_reset = (cond_sub_base > lower_bound) & (img<0.5)
    condition_set = (cond_sub_base < upper_bound) & (img>0.5)
    plot_cond(cond_sub_base,vmax=1000,title=f"set_tgv={int(tgv*10)},need_set={int(np.sum(condition_set))}")

In [ ]:
write_gray(img=img,write_times=20,start_tg_v=1,delta_tg_v=0.05,lower_bound=200,upper_bound=800,pulse_width=1e-6)